In [51]:
import h5py
import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import torchvision.transforms as transforms
import torchmetrics
import numpy as np
from tqdm import tqdm
import os
import pandas as pd

# Compute embeddings

In [111]:
class BaselineDataset(Dataset):
    def __init__(self, dataset_path, preprocessing, mode):
        super(BaselineDataset, self).__init__()
        self.dataset_path = dataset_path
        self.preprocessing = preprocessing
        self.mode = mode
        
        with h5py.File(self.dataset_path, 'r') as hdf:        
            self.image_ids = list(hdf.keys())

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        with h5py.File(self.dataset_path, 'r') as hdf:
            img = torch.tensor(hdf.get(img_id).get('img'))
            label = np.array(hdf.get(img_id).get('label')) if self.mode == 'train' else None
        return self.preprocessing(img).float(), label

In [105]:
def precompute(dataloader, model, device):
    xs, ys = [], []
    for x, y in tqdm(dataloader, leave=False):
        with torch.no_grad():
            xs.append(model(x.to(device)).detach().cpu().numpy())
        print(y)
        ys.append(y.numpy())
    xs = np.vstack(xs)
    ys = np.hstack(ys)
    return torch.tensor(xs), torch.tensor(ys)

In [101]:
class PrecomputedDataset(Dataset):
    def __init__(self, features, labels):
        super(PrecomputedDataset, self).__init__()
        self.features = features
        self.labels = labels.unsqueeze(-1)
    
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx].float()

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Working on {device}.')

Working on cpu.


In [10]:
feature_extractor = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device)
feature_extractor.eval()
linear_probing = torch.nn.Sequential(torch.nn.Linear(feature_extractor.num_features, 1),
                                     torch.nn.Sigmoid()).to(device)

Using cache found in C:\Users\Chardin Pierre/.cache\torch\hub\facebookresearch_dinov2_main
C:\Users\Chardin Pierre/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\Chardin Pierre/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\Chardin Pierre/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


In [103]:
def precompute_dataset(path, model, device, batch_size = 16, mode = 'train'):
    preprocessing = transforms.Resize((98, 98))
    dataset = BaselineDataset(path, preprocessing, mode)
    dataloader = DataLoader(dataset, shuffle=True, batch_size=batch_size)
    model.eval()
    return PrecomputedDataset(*precompute(dataloader, model, device))

In [22]:
for file in tqdm(os.listdir("data/train")):
    path = f"data/train/{file}"
    dataset = precompute_dataset(path, feature_extractor, device)
    torch.save(dataset, f"data/DINOv2_emb/train/"+file.replace(".h5", ".pth"))

100%|██████████| 1000/1000 [1:34:40<00:00,  5.68s/it]


In [23]:
for file in tqdm(os.listdir("data/val")):
    path = f"data/val/{file}"
    dataset = precompute_dataset(path, feature_extractor, device)
    torch.save(dataset, f"data/DINOv2_emb/val/"+file.replace(".h5", ".pth"))

100%|██████████| 1000/1000 [37:53<00:00,  2.27s/it]


# Linear probing

In [38]:
train_dataset = ConcatDataset(
    [torch.load(f"data/DINOv2_emb/train/{file}") for file in os.listdir("data/DINOv2_emb/train")])
val_dataset = ConcatDataset(
    [torch.load(f"data/DINOv2_emb/val/{file}") for file in os.listdir("data/DINOv2_emb/val")])

C:\Users\Chardin Pierre\AppData\Local\Temp\ipykernel_12012\1179852250.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  [torch.load(f"data/DINOv2_emb/train/{file}") for fi

In [40]:
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=16)
val_dataloader = DataLoader(val_dataset, shuffle=False, batch_size=16)

In [42]:
OPTIMIZER = 'Adam'
OPTIMIZER_PARAMS = {'lr': 0.001}
LOSS = 'BCELoss'
METRIC = 'Accuracy'
NUM_EPOCHS = 100
PATIENCE = 10

In [44]:
optimizer = getattr(torch.optim, OPTIMIZER)(linear_probing.parameters(), **OPTIMIZER_PARAMS)
criterion = getattr(torch.nn, LOSS)()
metric = getattr(torchmetrics, METRIC)('binary')
min_loss, best_epoch = float('inf'), 0

In [45]:
for epoch in range(NUM_EPOCHS):
    linear_probing.train()
    train_metrics, train_losses = [], []
    for train_x, train_y in tqdm(train_dataloader, leave=False):
        optimizer.zero_grad()
        train_pred = linear_probing(train_x.to(device))
        loss = criterion(train_pred, train_y.to(device))
        loss.backward()
        optimizer.step()
        train_losses.extend([loss.item()]*len(train_y))
        train_metric = metric(train_pred.cpu(), train_y.int().cpu())
        train_metrics.extend([train_metric.item()]*len(train_y))
    print(f'Epoch train [{epoch+1}/{NUM_EPOCHS}] | Loss {np.mean(train_losses):.4f} | Metric {np.mean(train_metrics):.4f}')

    linear_probing.eval()
    val_metrics, val_losses = [], []
    for val_x, val_y in tqdm(val_dataloader, leave=False):
        with torch.no_grad():
            val_pred = linear_probing(val_x.to(device))
        loss = criterion(val_pred, val_y.to(device))
        val_losses.extend([loss.item()]*len(val_y))
        val_metric = metric(val_pred.cpu(), val_y.int().cpu())
        val_metrics.extend([val_metric.item()]*len(val_y))
    print(f'Epoch valid [{epoch+1}/{NUM_EPOCHS}] | Loss {np.mean(val_losses):.4f} | Metric {np.mean(val_metrics):.4f}')

    if np.mean(val_losses) < min_loss:
        mean_val_loss = np.mean(val_losses)
        print(f'New best loss {min_loss:.4f} -> {mean_val_loss:.4f}')
        min_loss = mean_val_loss
        best_epoch = epoch
        torch.save(linear_probing.state_dict(), 'best_model.pth')

    if epoch - best_epoch == PATIENCE:
        break

Epoch train [1/100] | Loss 0.1772 | Metric 0.9326


Epoch valid [1/100] | Loss 0.3392 | Metric 0.8585
New best loss inf -> 0.3392


Epoch train [2/100] | Loss 0.1548 | Metric 0.9415


Epoch valid [2/100] | Loss 0.3548 | Metric 0.8562


Epoch train [3/100] | Loss 0.1500 | Metric 0.9437


...

Epoch valid [13/100] | Loss 0.3345 | Metric 0.8661


Epoch train [14/100] | Loss 0.1432 | Metric 0.9461


Epoch valid [14/100] | Loss 0.3479 | Metric 0.8713


Epoch train [15/100] | Loss 0.1436 | Metric 0.9462


Epoch valid [15/100] | Loss 0.3407 | Metric 0.8739


# Final predictions

To create a solutions file, you need to generate a CSV with 2 columns.
- **ID**: containing the ID of the image
- **Pred**: with the predicted class (**threshold the prediction to get either 0 or 1**)

In [98]:
torch.load("data/DINOv2_emb/test/test_0.pth").features.shape

C:\Users\Chardin Pierre\AppData\Local\Temp\ipykernel_12012\1148655247.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load("data/DINOv2_emb/test/test_0.pth").featu

torch.Size([86, 384])

In [47]:
linear_probing.load_state_dict(torch.load('DINO_baseline/best_model.pth', weights_only=True))
linear_probing.eval()
linear_probing.to(device)
prediction_dict = {}

In [112]:
def predict(model, path):
    solutions_data = {'ID': [], 'Pred': []}
    preprocessing = transforms.Resize((98, 98))

    with h5py.File(path, 'r') as hdf:
        for test_id in list(hdf.keys()):
            img = preprocessing(torch.tensor(np.array(hdf.get(test_id).get('img')))).unsqueeze(0).float()
            pred = model(feature_extractor(img.to(device))).detach().cpu()
            solutions_data['ID'].append(int(test_id))
            solutions_data['Pred'].append(int(pred.item() > 0.5))
    return solutions_data['ID'], solutions_data['Pred']

In [113]:
predictions = [predict(linear_probing, f"data/test/{file}") for file in tqdm(os.listdir("data/test"))]

100%|██████████| 1000/1000 [5:44:47<00:00, 20.69s/it] 


In [114]:
solutions_data = pd.DataFrame({'ID': np.concatenate([np.array(i) for i,_ in predictions]),
                                'Pred': np.concatenate([np.array(p) for _,p in predictions])})\
                        .set_index('ID')\
                        .sort_index()

In [115]:
solutions_data.to_csv('DINO_baseline/baseline.csv')

In [116]:
solutions_data

,Pred
ID,
0,0
1,0
2,0
3,1
4,0
...,...
85049,0
85050,1
85051,0
